In [1]:
import ipywidgets as widgets
from IPython.display import display
import ipycanvas

# --- Инициализация виджетов ---
editor = widgets.Textarea(
    value="",
    placeholder="Введите программу здесь...",
    description="Редактор:",
    layout=widgets.Layout(width="100%", height="200px"),
)

terminal = widgets.Text(
    value="",
    placeholder="Введите команду...",
    description="Терминал:",
    layout=widgets.Layout(width="100%"),
)

terminal_output = widgets.Textarea(
    value="",
    description="Вывод:",
    layout=widgets.Layout(width="100%", height="100px"),
    disabled=True,
)

execute_button = widgets.Button(description="Выполнить", layout=widgets.Layout(width="150px"))
clear_button = widgets.Button(description="Очистить", layout=widgets.Layout(width="150px"))

canvas = ipycanvas.Canvas(width=600, height=400, layout=widgets.Layout(width="100%"))

# --- Состояние программы ---
points = {}
macros = {}
terminal_history = []

# --- Функции для работы с макросами ---
def define_macro(name: str, params: list, body: list):
    macros[name] = {"params": params, "body": body}

def substitute_macro(macro_name: str, actual_params: list) -> list:
    macro = macros[macro_name]
    substituted_body = []
    param_mapping = dict(zip(macro["params"], actual_params))

    for command in macro["body"]:
        substituted_command = command
        for param, actual in param_mapping.items():
            substituted_command = substituted_command.replace(param, actual)
        substituted_body.append(substituted_command)

    return substituted_body

# --- Функции для парсинга и исполнения ---
def parse_and_execute(command: str):
    parts = command.split()
    if not parts:
        return

    cmd_type = parts[0]

    if cmd_type == "&":
        macro_name = parts[1]
        params = []
        body = []
        i = 2
        while i < len(parts) and parts[i].startswith("$"):
            params.append(parts[i])
            i += 1
        body = parts[i:]
        define_macro(macro_name, params, body)
        terminal_history.append(f"Макрос {macro_name} определён с параметрами {params}")
    elif cmd_type == "!":
        macro_name = parts[1]
        actual_params = parts[2:]
        expanded_commands = substitute_macro(macro_name, actual_params)
        for cmd in expanded_commands:
            parse_and_execute(cmd)
    elif cmd_type == ".":
        point_name = parts[1]
        x, y = 100 + len(points) * 50, 100 + len(points) * 30
        points[point_name] = (x, y)
        draw_point(point_name, x, y)
        terminal_history.append(f"Создана точка {point_name}")
    elif cmd_type == "=":
        segment_name = parts[1]
        point1 = parts[2]
        point2 = parts[3]
        if point1 in points and point2 in points:
            x1, y1 = points[point1]
            x2, y2 = points[point2]
            draw_segment(segment_name, x1, y1, x2, y2)
            terminal_history.append(f"Создан отрезок {segment_name} между {point1} и {point2}")
        else:
            terminal_history.append(f"Ошибка: точки {point1} или {point2} не существуют")
    elif cmd_type == "@":
        circle_name = parts[1]
        center = parts[2]
        point_on_circle = parts[3]
        if center in points and point_on_circle in points:
            cx, cy = points[center]
            px, py = points[point_on_circle]
            radius = ((px - cx) ** 2 + (py - cy) ** 2) ** 0.5
            draw_circle(circle_name, cx, cy, radius)
            terminal_history.append(f"Создана окружность {circle_name} с центром в {center} и точкой {point_on_circle}")
        else:
            terminal_history.append(f"Ошибка: точки {center} или {point_on_circle} не существуют")
    elif cmd_type == "+":
        if "," in parts[1]:
            point_names = parts[1].split(",")
            circle1 = parts[2]
            circle2 = parts[3]
            terminal_history.append(f"Найдены точки пересечения {point_names} окружностей {circle1} и {circle2}")
        else:
            point_name = parts[1]
            segment1 = parts[2]
            segment2 = parts[3]
            terminal_history.append(f"Найдена точка пересечения {point_name} отрезков {segment1} и {segment2}")
    else:
        terminal_history.append(f"Неизвестная команда: {command}")

# --- Функции для рисования ---
def draw_point(name: str, x: float, y: float):
    with canvas.hold_sync():
        canvas.fill_style = "red"
        canvas.fill_arc(x, y, 5, 0, 2 * 3.14)
        canvas.fill_text(name, x + 10, y - 10)

def draw_segment(name: str, x1: float, y1: float, x2: float, y2: float):
    with canvas.hold_sync():
        canvas.stroke_style = "blue"
        canvas.begin_path()
        canvas.move_to(x1, y1)
        canvas.line_to(x2, y2)
        canvas.stroke()
        canvas.fill_text(name, (x1 + x2) / 2, (y1 + y2) / 2 - 10)

def draw_circle(name: str, center_x: float, center_y: float, radius: float):
    with canvas.hold_sync():
        canvas.stroke_style = "green"
        canvas.begin_path()
        canvas.arc(center_x, center_y, radius, 0, 2 * 3.14)
        canvas.stroke()
        canvas.fill_text(name, center_x + 10, center_y - 10)

# --- Функция для обновления вывода терминала ---
def update_terminal():
    terminal_output.value = "\n".join(terminal_history[-10:])

# --- Обработчики событий ---
def on_execute_button_click(b):
    code = editor.value
    for line in code.split("\n"):
        line = line.strip()
        if line:
            parse_and_execute(line)
    update_terminal()

def on_clear_button_click(b):
    global points, macros, terminal_history
    points = {}
    macros = {}
    terminal_history = []
    canvas.clear()
    editor.value = ""
    terminal.value = ""
    terminal_output.value = ""

def on_terminal_submit(change):
    if change["type"] == "change" and change["name"] == "value":
        command = terminal.value.strip()
        if command:
            parse_and_execute(command)
            terminal.value = ""
            update_terminal()

execute_button.on_click(on_execute_button_click)
clear_button.on_click(on_clear_button_click)
terminal.observe(on_terminal_submit)

# --- Инициализация макросов по умолчанию ---
define_macro(
    "midpoint",
    ["$1", "$2", "$3"],
    [
        "= ab $2 $3",
        "@ Ca $2 $3",
        "@ Cb $3 $2",
        "+ p1,p2 Ca Cb",
        "= p1p2 p1 p2",
        "+ $1 ab p1p2"
    ]
)
terminal_history.append("Макрос midpoint загружен по умолчанию")

# --- Сборка интерфейса ---
editor_box = widgets.VBox([editor])
terminal_box = widgets.VBox([terminal, terminal_output])
buttons_box = widgets.HBox([execute_button, clear_button])
canvas_box = widgets.VBox([canvas])

ui = widgets.VBox([
    editor_box,
    terminal_box,
    buttons_box,
    canvas_box,
])

display(ui)
update_terminal()

IndexError: list index out of range

IndexError: list index out of range

IndexError: list index out of range